# Musanga — Block 13B: native PlanetScope / UAV dense training tiles

This is a **standalone Colab notebook** (avoids stale NB06 copies). It uses the already computed full-resolution UAV U-Net probability maps as **dense supervision**, and annual PlanetScope blue/green/red/NIR reflectance as predictors. It **does not** read original UAV RGB orthomosaics, rerun U-Net inference, or retrain earlier regressions.

### Stage 13B goals

- Create non-overlapping **128×128 Planet pixels** (~611×611 m) on each native Planet raster grid, anchored to the existing verified 50 m ROI samples.
- Warp **only each tile's footprint** from UAV uint16 probability maps by area averaging; decode using confirmed **/10000**. Preserve separate ROI, satellite-valid, and UAV-valid masks; do not use padded/nodata pixels as negative examples.
- Save **georeferenced tile metadata**, source names, valid fractions, geographic footprint and fixed tile IDs. Split assignment belongs to *13C*, where buffers must be based on tile extents and not just label centers; the two MOS mosaics stay in one forest holdout.
- Cached chunks permit safe restart. Default **three-tile smoke test**; inspect output before large-scale extraction.

**Important limitation:** these are teacher-supervised spatial predictions. Pixel-level probability agreement with the UAV model is *not* independent ground truth. Keeping tile footprints will allow rigorous geographic holdouts later. Monthly Planet/S2/embedding channels are *not* mixed into this first dataset; they will be added through independent branches after a Planet-only dense U-Net baseline.

Run the code cell; first run writes only 3 new tiles (`MAX_NEW_TILES13B=3`). Once we inspect its outputs, change that parameter to `None` to finish cached extraction.

In [ ]:
# 13B — Restartable native Planet/UAV dense training tiles.
from pathlib import Path
import os,json,hashlib,time,math
import numpy as np,pandas as pd,rasterio,geopandas as gpd
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import array_bounds
from pyproj import Transformer
from shapely.geometry import box
from IPython.display import display

ROOTS13B=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
          Path('/content/drive/MyDrive/Musanga/Final')]
FINAL13B=next((p for p in ROOTS13B if p.is_dir()),None)
if FINAL13B is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL13B=next((p for p in ROOTS13B if p.is_dir()),None)
assert FINAL13B is not None,'Musanga Final workspace not found.'
ROOT13B=FINAL13B/'Outputs/V3_1'
NB05_13B=ROOT13B/'05_sat_predictor_exploration'
OUT13B=ROOT13B/'06_spatial_planet_embeddings/13_dense_planet_unet/13B_tiles'
OUT13B.mkdir(parents=True,exist_ok=True)
SRC_UAV13B=ROOT13B/'03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
ANCHOR_FILE13B=ROOT13B/'06_spatial_planet_embeddings/09_uav_targets_50m.parquet'
MANIFEST_FILE13B=ROOT13B/'00_input_audit/00_input_manifest.json'
assert ANCHOR_FILE13B.is_file() and MANIFEST_FILE13B.is_file()
ANCH13B=pd.read_parquet(ANCHOR_FILE13B)
assert {'site','mosaic','x','y','crs','mean_probability'}.issubset(ANCH13B.columns)
manifest=json.loads(MANIFEST_FILE13B.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.exists(),roi_file
ROI13B=gpd.read_file(roi_file)
assert not ROI13B.empty and ROI13B.crs

YRS13B={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
MOSAICS13B={'Baego':['Baego.tif'],
            'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif',
                   'Mos_5_6_transparent_mosaic_group1_georef.tif'],
            'Yangambi':['Yangambi_georef.tif'],
            'Yoko':['Yoko_kvo_georef.tif']}
SIZE13B=128
MAX_NEW_TILES13B=3    # First test: 3. After inspection, set None.
MIN_ROI13B=.70         # Candidate tile intersects >=70% ROI.
MIN_VALID_PIXELS13B=.70
MIN_PLANET13B=.90
SRC_BANDS13B=[13,14,15,16]  # B,G,R,N
ENCODING13B=10000.0
def planet13b(site):
    d=NB05_13B/'input_layers/Musanga_V31_NB05_UAV_ROI_Annual'
    matches=sorted(d.glob(f'NB05_UAV_ROI_{site}_{YRS13B[site]}_Annual_19band*.tif'))
    assert len(matches)==1,(site,matches)
    return matches[0]
def teacher13b(mosaic):
    f=SRC_UAV13B/f'{Path(mosaic).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
    assert f.exists(),f
    return f
sources13b={site:planet13b(site) for site in YRS13B}
config13b=dict(version='dense_planet_teacher_v1',tile_px=SIZE13B,
               encoding=ENCODING13B,bands=SRC_BANDS13B,
               min_roi=MIN_ROI13B,min_valid=MIN_VALID_PIXELS13B,
               min_planet=MIN_PLANET13B,
               planet={k:(str(v),v.stat().st_size,v.stat().st_mtime_ns)
                       for k,v in sources13b.items()},
               teacher={m:(str(teacher13b(m)),teacher13b(m).stat().st_size,
                           teacher13b(m).stat().st_mtime_ns)
                        for vv in MOSAICS13B.values() for m in vv},
               anchors=(str(ANCHOR_FILE13B),ANCHOR_FILE13B.stat().st_size,
                        ANCHOR_FILE13B.stat().st_mtime_ns),
               roi=(str(roi_file),roi_file.stat().st_size,roi_file.stat().st_mtime_ns))
RUNID13B=hashlib.sha256(json.dumps(config13b,sort_keys=True).encode()).hexdigest()[:16]
RUN13B=OUT13B/RUNID13B
RUN13B.mkdir(parents=True,exist_ok=True)
conf_file=RUN13B/'config.json'
if conf_file.exists():
    assert json.loads(conf_file.read_text())==config13b,'Cached source/config mismatch.'
else:
    conf_file.write_text(json.dumps(config13b,indent=2))
print('13B cache:',RUN13B,flush=True)

# Snap canonical 50 m ROI-approved anchors to non-overlapping Planet tiles.
# The Planet raster origin defines the lattice, NOT per-mosaic UAV origins.
records=[]
for site,mosaics in MOSAICS13B.items():
    with rasterio.open(sources13b[site]) as planet:
        assert planet.count>=16 and planet.crs and 4.<abs(planet.res[0])<5.5
        for mosaic in mosaics:
            sub=ANCH13B[(ANCH13B.site==site)&(ANCH13B.mosaic==mosaic)]
            assert len(sub)>0,(site,mosaic)
            assert sub.crs.nunique()==1
            tf=Transformer.from_crs(sub.crs.iloc[0],planet.crs,always_xy=True)
            x,y=tf.transform(sub.x.to_numpy(float),sub.y.to_numpy(float))
            px,py=(~planet.transform)*(x,y)
            # Reproducible globally aligned 128-pixel tile origin.
            col0=(np.floor(px/SIZE13B).astype('int64')*SIZE13B)
            row0=(np.floor(py/SIZE13B).astype('int64')*SIZE13B)
            unique=sorted(set(zip(row0,col0)))
            for row,col in unique:
                if row<0 or col<0 or row+SIZE13B>planet.height or col+SIZE13B>planet.width:
                    continue
                win=Window(int(col),int(row),SIZE13B,SIZE13B)
                transform=planet.window_transform(win)
                west,south,east,north=array_bounds(SIZE13B,SIZE13B,transform)
                records.append(dict(site=site,mosaic=mosaic,
                    row0=int(row),col0=int(col),
                    tile_id=f'{site}_{Path(mosaic).stem}_r{row:06d}_c{col:06d}',
                    crs=str(planet.crs),west=west,south=south,east=east,north=north,
                    centre_x=(west+east)/2,centre_y=(south+north)/2,
                    anchor_count=int(((row0==row)&(col0==col)).sum())))
TILES13B=pd.DataFrame(records).sort_values(['site','mosaic','row0','col0']).reset_index(drop=True)
assert len(TILES13B) and not TILES13B.tile_id.duplicated().any()
TILES13B.to_parquet(RUN13B/'tile_manifest_all_candidates.parquet',index=False)
print('Candidate tiles by mosaic:')
display(TILES13B.groupby(['site','mosaic']).agg(
    candidate_tiles=('tile_id','size'),anchors=('anchor_count','sum')))

# Extract windows per mosaic; no full-resolution UAV RGB/teacher raster arrays.
new=0
for (site,mosaic),grp in TILES13B.groupby(['site','mosaic'],sort=True):
    tile_dir=RUN13B/site/Path(mosaic).stem
    tile_dir.mkdir(parents=True,exist_ok=True)
    with rasterio.open(sources13b[site]) as planet, rasterio.open(teacher13b(mosaic)) as teacher:
        assert teacher.crs and teacher.nodata==65535,(mosaic,'unexpected teacher nodata',teacher.nodata)
        # Repair the earlier implicit VRT nodata conversion issue explicitly.
        with WarpedVRT(teacher,crs=planet.crs,transform=planet.transform,
                       width=planet.width,height=planet.height,
                       src_nodata=65535,nodata=65535,
                       dtype='float32',resampling=Resampling.average,
                       warp_mem_limit=64) as aligned:
            roi_local=ROI13B.to_crs(planet.crs)
            roi_geom=[g for g in roi_local.geometry if g is not None and not g.is_empty]
            assert roi_geom
            for rec in grp.itertuples(index=False):
                dest=tile_dir/f'{rec.tile_id}.npz'
                if dest.exists():
                    try:
                        with np.load(dest,allow_pickle=False) as old:
                            assert old['planet'].shape==(4,SIZE13B,SIZE13B)
                            assert old['probability'].shape==(SIZE13B,SIZE13B)
                            assert old['valid'].shape==(SIZE13B,SIZE13B)
                        continue
                    except Exception as err:
                        print('REBUILD partial',dest.name,str(err)[:100],flush=True)
                if MAX_NEW_TILES13B is not None and new>=MAX_NEW_TILES13B:break
                start=time.monotonic()
                win=Window(rec.col0,rec.row0,SIZE13B,SIZE13B)
                ptfm=planet.window_transform(win)
                image=planet.read(SRC_BANDS13B,window=win,out_dtype='float32')
                pv=planet.dataset_mask(window=win)>0
                pv &= np.isfinite(image).all(axis=0)&(image>0).all(axis=0)
                teacher_raw=aligned.read(1,window=win,out_dtype='float32')
                tm=aligned.dataset_mask(window=win)>0
                tv=tm&np.isfinite(teacher_raw)&(teacher_raw>=0)&(teacher_raw<=10000)
                roim=rasterize(((g,1) for g in roi_geom),out_shape=(SIZE13B,SIZE13B),
                               transform=ptfm,dtype='uint8',all_touched=False)>0
                valid=roim&pv&tv
                fraction=dict(roi=float(roim.mean()),planet=float(pv.mean()),
                              teacher=float(tv.mean()),joint=float(valid.mean()))
                accepted=(fraction['roi']>=MIN_ROI13B and
                          fraction['planet']>=MIN_PLANET13B and
                          fraction['joint']>=MIN_VALID_PIXELS13B)
                prob=np.clip(teacher_raw/ENCODING13B,0,1).astype('float32')
                prob[~valid]=0.
                image[:,~pv]=0.
                metadata=dict(site=site,mosaic=mosaic,tile_id=rec.tile_id,
                    crs=rec.crs,row0=int(rec.row0),col0=int(rec.col0),
                    bounds=[rec.west,rec.south,rec.east,rec.north],
                    anchor_count=int(rec.anchor_count),
                    roi_fraction=fraction['roi'],planet_fraction=fraction['planet'],
                    teacher_fraction=fraction['teacher'],joint_fraction=fraction['joint'],
                    accepted=bool(accepted),probability_encoding='uint16_div_10000',
                    source_type='UAV_U-Net_pseudolabel',size=SIZE13B)
                tmp=dest.with_suffix('.npz.partial')
                with open(tmp,'wb') as f:
                    np.savez_compressed(f,planet=np.clip(image,-65000,65000).astype('float16'),
                                        probability=prob.astype('float16'),
                                        valid=valid.astype('uint8'),
                                        roi=roim.astype('uint8'),
                                        metadata=json.dumps(metadata))
                os.replace(tmp,dest)
                new+=1
                print('13B SAVED',rec.tile_id,
                      'accepted',accepted,'valid',round(fraction['joint'],3),
                      'seconds',round(time.monotonic()-start,2),flush=True)
            if MAX_NEW_TILES13B is not None and new>=MAX_NEW_TILES13B:break
    if MAX_NEW_TILES13B is not None and new>=MAX_NEW_TILES13B:break

status=[]
for row in TILES13B.itertuples(index=False):
    path=RUN13B/row.site/Path(row.mosaic).stem/f'{row.tile_id}.npz'
    base=row._asdict()
    base['saved']=path.exists()
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            meta=json.loads(str(z['metadata'].item()))
        for key in ('accepted','joint_fraction','roi_fraction','planet_fraction','teacher_fraction'):
            base[key]=meta[key]
    status.append(base)
STATE13B=pd.DataFrame(status)
STATE13B.to_csv(RUN13B/'tile_manifest_progress.csv',index=False)
print('13B saved',int(STATE13B.saved.sum()),'of',len(STATE13B),
      'candidate tiles, accepted so far',
      int(STATE13B.accepted.fillna(False).sum()),flush=True)
display(STATE13B[STATE13B.saved].tail(12))
print('13B tile cache:',RUN13B)
print('Keep MAX_NEW_TILES13B=3 until these pilot tiles have been reviewed.')
